# Forward-Forward MNIST classifier — ECEN 726 Project #5 (Group 2)

This notebook trains a fully connected network with Hinton's **Forward-Forward (FF)** algorithm
(Hinton 2022) and compares it with **backpropagation (BP)** using the same ReLU hidden layers.

All algorithm code lives in `src/` (NumPy only, no ML frameworks); this notebook only calls it and plots:

| Module | Contents |
|---|---|
| `src/mnist.py` | MNIST download, 50k / 10k / 10k train / val / test split |
| `src/ff.py` | label embedding, negatives, FF layer (loss + hand-derived gradient), `FFNetwork`, `train_ff` |
| `src/bp_relu.py` | BP baseline: ReLU hidden layers + sigmoid output, `backprop_relu`, `ReLUCrossbarNetwork` |
| `src/crossbar.py` | memristor crossbar (G+/G− pair, 0.1–150 µS, σ = 0.1 µS) — used in Phase 3 |

**Protocol:** hyperparameters were chosen on the **validation** set; the **test** set is evaluated once per final model.

Set `RETRAIN = True` to retrain everything from scratch (FF ≈ 7 min, BP ≈ 2 min, random-vs-hard ≈ 5 min).
With `RETRAIN = False` the notebook loads the saved weights and training histories from `../results/`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

from src.mnist import load_mnist
from src.ff import (FFNetwork, train_ff, embed_label, random_wrong_labels,
                    normalize, layer_loss_and_grads, check_layer_gradients)
from src.bp_relu import ReLUNetwork, backprop_relu, check_gradients

RETRAIN = False                        # True: retrain all models (~15 min); False: load saved results
RESULTS_DIR = Path("..") / "results"   # outside the git repo, so weights are never committed
RESULTS_DIR.mkdir(exist_ok=True)

# Shared settings of the final models (chosen on the validation set)
LAYERS = [784, 500, 500]
FF_CFG = dict(epochs=60, lr=0.003, optimizer="adam", batch_size=100, negatives="hard")
THETA = 5.0
BP_CFG = dict(epochs=60, lr=0.1, batch_size=32)

np.seterr(over="ignore")  # sigmoid of very negative z overflows harmlessly to 0

def style(ax, xlabel, ylabel, title):
    """Same plot style as main.ipynb."""
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(frameon=False); ax.grid(True, color="0.9")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

## 1. Data and label embedding

FF has no output layer, so the class has to enter through the input. Following Hinton (Sec. 3.3), the
first 10 pixels of the flattened image (top-left of the black border, always 0 in MNIST) are overwritten
with a one-hot label:

* **positive** example: image + correct label → the network should make its goodness **high**
* **negative** example: the *same* image + a wrong label → goodness should be **low**
* **neutral** label: 0.1 in all 10 pixels (for when the label is unknown)

Because positive and negative differ only in the label pixels, the net can only tell them apart by
learning which image features go with which label.

In [ ]:
splits = load_mnist()
X_train, y_train = splits["train"]
X_val, y_val = splits["val"]
X_test, y_test = splits["test"]
for name, (X, y) in splits.items():
    print(f"{name:>5}: X {X.shape}, y {y.shape}")
print("max of the first 10 pixels over all images:",
      max(X.reshape(len(X), -1)[:, :10].max() for X, _ in splits.values()))

In [ ]:
i = 7
rng = np.random.default_rng(0)
wrong = random_wrong_labels(y_train[i:i + 1], rng)
versions = [("positive", embed_label(X_train[i:i + 1], y_train[i:i + 1]), f"label {y_train[i]} (correct)"),
            ("negative", embed_label(X_train[i:i + 1], wrong), f"label {wrong[0]} (wrong)"),
            ("neutral", embed_label(X_train[i:i + 1], None), "0.1 everywhere")]

fig, axes = plt.subplots(2, 3, figsize=(9, 4.2), gridspec_kw={"height_ratios": [4, 1]})
for col, (name, Xe, desc) in enumerate(versions):
    axes[0, col].imshow(Xe.reshape(28, 28), cmap="gray", vmin=0, vmax=1)
    axes[0, col].set_title(f"{name}: {desc}", fontsize=10)
    axes[0, col].axis("off")
    # Zoom on the label pixels: the first 10 entries of the flattened image
    axes[1, col].imshow(Xe[:, :10], cmap="gray", vmin=0, vmax=1, aspect="auto")
    axes[1, col].set_xticks(range(10)); axes[1, col].set_yticks([])
    axes[1, col].set_xlabel("label pixels 0-9", fontsize=9)
plt.tight_layout(); plt.show()

## 2. One FF layer and its local gradient

For a layer input $x$ (already normalized to unit length):

$$z = Wx + b,\qquad y = \mathrm{ReLU}(z),\qquad G = \sum_j y_j^2,\qquad p = \sigma(G - \theta)$$

The local loss pushes positive goodness above the threshold $\theta$ and negative goodness below it:

$$L = \mathrm{softplus}\big(-(G_{pos}-\theta)\big) + \mathrm{softplus}\big(G_{neg}-\theta\big)$$

Gradient, derived by hand (no autograd):

$$\frac{\partial L}{\partial G_{pos}} = -(1-p_{pos}),\quad
\frac{\partial L}{\partial G_{neg}} = p_{neg},\quad
\delta = \frac{\partial L}{\partial G}\cdot 2y,\quad
\nabla W = \delta_{pos}x_{pos}^T + \delta_{neg}x_{neg}^T$$

The weight gradient is an **outer product of the layer's own input and a local error** — no $W^T$ and no
error from other layers. That is what makes FF attractive for crossbars (no weight transport).

Between layers only the direction $y/\lVert y\rVert$ is passed on; otherwise the next layer could read
off this layer's goodness from the vector length and learn nothing new.

Before training, the hand-derived gradients are checked against central finite differences.

In [ ]:
rng = np.random.default_rng(1)
B = 8
x_pos = normalize(embed_label(X_train[:B], y_train[:B]).T)                          # (784, B)
x_neg = normalize(embed_label(X_train[:B], random_wrong_labels(y_train[:B], rng)).T)
for n_out, theta in [(20, 2.0), (50, 5.0), (100, 5.0)]:
    W = rng.standard_normal((n_out, 784)) * np.sqrt(2 / n_out) * 3
    b = rng.standard_normal((n_out, 1)) * 0.1
    err = check_layer_gradients(W, b, x_pos, x_neg, theta)
    print(f"FF layer 784->{n_out}, theta={theta}: max relative error {err:.1e}")

err = check_gradients(ReLUNetwork([784, 16, 16, 10], seed=1),
                      X_train[:B].astype(np.float64), y_train[:B])
print(f"BP-ReLU 784-16-16-10:          max relative error {err:.1e}")

## 3. Train the FF network (784-500-500)

* every batch: positive and negative versions go forward; **each layer updates from its own loss only**
  and passes normalized activities to the next layer
* optimizer: Adam (lr 0.003). Plain SGD only worked in a narrow lr range (lr ≥ 1 killed every ReLU)
* threshold θ = 5: θ = 1 / 2 / 5 gave 95.9 / 96.5 / 97.4 % val accuracy after 10 epochs; Hinton's
  θ = #neurons saturates the loss with unit-length inputs
* **hard negatives**: from epoch 2, the wrong label is sampled in proportion to how much the net
  currently likes it (softmax of its goodness, true label excluded)
* prediction: embed each of the 10 labels, sum goodness over layers, take the argmax

`test_acc` in the training history below is measured on the **validation** set.

In [ ]:
ff_weights = RESULTS_DIR / "ff_784_500_500.npz"
final_json = RESULTS_DIR / "final_results.json"
saved = json.loads(final_json.read_text()) if final_json.exists() else {}

ff_net = FFNetwork(LAYERS, theta=THETA)
if RETRAIN or not ff_weights.exists():
    ff_net, ff_hist = train_ff(ff_net, X_train, y_train, **FF_CFG, X_test=X_val, y_test=y_val)
    np.savez(ff_weights, *ff_net.weights, *ff_net.biases, theta=np.array(ff_net.theta))
    saved["ff"] = dict(theta=THETA, history={k: ff_hist[k] for k in ("loss", "train_acc", "test_acc")})
    final_json.write_text(json.dumps(saved, default=float))
else:
    z = np.load(ff_weights)
    n = len(ff_net.weights)
    ff_net.weights = [z[f"arr_{l}"] for l in range(n)]
    ff_net.biases = [z[f"arr_{n + l}"] for l in range(n)]
    ff_hist = saved["ff"]["history"]
    print(f"loaded {ff_weights}  ({ff_net})")
print(f"FF final: train {ff_hist['train_acc'][-1]:.4f}, val {ff_hist['test_acc'][-1]:.4f}")

## 4. BP baseline with the same hidden layers (784-500-500-10)

Same ReLU hidden layers as FF, plus a 10-unit sigmoid output layer trained with cross-entropy
(the same loss as the group's `backprop.py`). Errors are sent backwards through $W^T$ — the step FF avoids.

In [ ]:
if RETRAIN or "bp" not in saved:
    bp_net, bp_hist = backprop_relu(ReLUNetwork(LAYERS + [10]), X_train, y_train, **BP_CFG,
                                    X_test=X_val, y_test=y_val)
    saved["bp"] = dict(history=bp_hist, test=float((bp_net.predict(X_test) == y_test).mean()))
    final_json.write_text(json.dumps(saved, default=float))
else:
    bp_net = None   # weights were not saved; set RETRAIN = True to rebuild the BP net
    bp_hist = saved["bp"]["history"]
    print("loaded BP history from", final_json)
print(f"BP final: train {bp_hist['train_acc'][-1]:.4f}, val {bp_hist['test_acc'][-1]:.4f}")

## 5. Test accuracy (evaluated once per final model)

In [ ]:
ff_test = (ff_net.predict(X_test) == y_test).mean()
bp_test = (bp_net.predict(X_test) == y_test).mean() if bp_net is not None else saved["bp"]["test"]
print(f"{'model':<28}{'val':>8}{'test':>8}")
print(f"{'FF   784-500-500':<28}{ff_hist['test_acc'][-1]:>8.4f}{ff_test:>8.4f}")
print(f"{'BP   784-500-500-10 (ReLU)':<28}{bp_hist['test_acc'][-1]:>8.4f}{bp_test:>8.4f}")

## 6. FF vs BP: accuracy per epoch

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for hist, name, color in [(ff_hist, "FF", "#2a6fdb"), (bp_hist, "BP", "#e8710a")]:
    ep = np.arange(1, len(hist["test_acc"]) + 1)
    ax.plot(ep, hist["test_acc"], color=color, linewidth=2, label=f"{name} validation")
    ax.plot(ep, hist["train_acc"], color=color, linewidth=1, linestyle="--", label=f"{name} train")
    ax.annotate(f"{name} {hist['test_acc'][-1]:.4f}", (ep[-1], hist["test_acc"][-1]),
                xytext=(5, 0), textcoords="offset points", va="center", color="0.25")
ax.set_ylim(0.9, 1.001)
style(ax, "Epoch", "Accuracy", "Forward-Forward vs backprop, 784-500-500 ReLU")
plt.show()

## 7. Random vs hard negatives

Same network and settings except θ = 2, 20 epochs (this comparison was run before θ was tuned).
Hard negatives focus training on the labels the net still confuses.

In [ ]:
neg_json = RESULTS_DIR / "neg_compare.json"
if RETRAIN or not neg_json.exists():
    neg = {}
    for kind in ("random", "hard"):
        _, h = train_ff(FFNetwork(LAYERS, theta=2.0), X_train, y_train, epochs=20, lr=0.003,
                        negatives=kind, X_test=X_val, y_test=y_val)
        neg[kind] = dict(loss=h["loss"], train_acc=h["train_acc"], val_acc=h["test_acc"])
    neg_json.write_text(json.dumps(neg, default=float))
else:
    neg = json.loads(neg_json.read_text())

fig, ax = plt.subplots(figsize=(8, 4))
for kind, color in [("random", "0.5"), ("hard", "#2a6fdb")]:
    ep = np.arange(1, len(neg[kind]["val_acc"]) + 1)
    ax.plot(ep, neg[kind]["val_acc"], color=color, linewidth=2, label=f"{kind} negatives")
    ax.annotate(f"{neg[kind]['val_acc'][-1]:.4f}", (ep[-1], neg[kind]["val_acc"][-1]),
                xytext=(5, 0), textcoords="offset points", va="center", color="0.25")
style(ax, "Epoch", "Validation accuracy", "Random vs hard negatives (θ = 2)")
plt.show()

## 8. Goodness of positive vs negative data, per layer

After training, each layer should give positive inputs (correct label) goodness above θ and negative
inputs (wrong label) goodness below it. The overlap of the two histograms is where errors come from.

In [ ]:
rng = np.random.default_rng(0)
Xs, ys = X_val[:2000], y_val[:2000]
_, G_pos = ff_net.layer_outputs(embed_label(Xs, ys))
_, G_neg = ff_net.layer_outputs(embed_label(Xs, random_wrong_labels(ys, rng)))

fig, axes = plt.subplots(1, len(G_pos), figsize=(11, 3.8))
for l, ax in enumerate(axes):
    hi = np.percentile(np.concatenate([G_pos[l], G_neg[l]]), 99)
    bins = np.linspace(0, hi, 60)
    ax.hist(G_neg[l], bins=bins, color="0.6", alpha=0.7, label="negative (wrong label)")
    ax.hist(G_pos[l], bins=bins, color="#2a6fdb", alpha=0.7, label="positive (correct label)")
    ax.axvline(ff_net.theta[l], color="#e8710a", linewidth=2, label=f"θ = {ff_net.theta[l]:g}")
    sep = (G_pos[l] > ff_net.theta[l]).mean() / 2 + (G_neg[l] < ff_net.theta[l]).mean() / 2
    style(ax, "Goodness G = Σ y²", "Count", f"Layer {l + 1}: {sep:.1%} on the right side of θ")
plt.tight_layout(); plt.show()

## 9. Next: FF on the memristor crossbar

* **Ex-situ:** program the trained `ff_net` weights onto `Crossbar` layers (0.1–150 µS, G+/G− pair,
  σ = 0.1 µS programming noise) and run the FF forward pass and goodness-based prediction on the chip.
  Unit-length layer inputs keep every row voltage ≤ V_READ = 0.2 V.
* **In-situ (if required):** keep the weights as conductances during training and reprogram after each
  local outer-product update.